# TSLA Momentum Model — Notebook 2: Backtest Engine

Runs the signals from Notebook 1 through an event-driven backtest: trades execute at the next day's open (never the signal day's close), positions are sized off a fixed risk-per-trade rule rather than going all-in, commissions and slippage are applied, and a stop-loss is checked against the day's *low* rather than only the close, so a stop that's breached intraday isn't missed.

See `src/backtest.py` for the full mechanics and the rationale behind each assumption.

In [ ]:
import sys
import os

# Resolve the src/ folder regardless of whether the working directory is
# the notebook's own folder (Jupyter/Colab default) or the project root
# (which some IDEs, including PyCharm's Jupyter integration, may use instead).
for _candidate in ("../src", "src"):
    if os.path.isdir(_candidate):
        sys.path.insert(0, _candidate)
        break
else:
    raise RuntimeError(
        "Could not locate the src/ folder from the current working directory: "
        + os.getcwd()
    )

import pandas as pd
import matplotlib.pyplot as plt

from backtest import run_backtest, trades_to_frame

df = pd.read_pickle("tsla_signals.pkl")
df.tail()

## Backtest parameters

- **Initial capital:** $100,000
- **Risk per trade:** 1% of current cash, sized by distance from entry to the ATR-based stop
- **Stop distance:** 2.5 × ATR(14) below entry
- **Commission:** $0.005/share (in line with typical US equity retail commission schedules)
- **Slippage:** 5 bps applied to both entries and exits

These aren't tuned to flatter the backtest — they're standard, defensible assumptions, and Notebook 3 stress-tests sensitivity to a couple of them.

In [ ]:
equity_curve, trades = run_backtest(
    df,
    initial_capital=100_000.0,
    risk_pct=0.01,
    atr_multiplier=2.5,
    commission_per_share=0.005,
    slippage_bps=5.0,
)

trade_log = trades_to_frame(trades)
print(f"Number of trades: {len(trade_log)}")
trade_log

## Equity curve, head and tail

In [ ]:
equity_curve.head()

In [ ]:
equity_curve.tail()

## Quick look at strategy vs. buy-and-hold

Full metrics and the in-sample/out-of-sample robustness check are in Notebook 3 — this is just a first-pass visual.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(equity_curve.index, equity_curve["equity"], label="Strategy")
ax.plot(equity_curve.index, equity_curve["buy_hold_equity"], label="Buy & Hold TSLA", linestyle="--")
ax.set_title("Equity Curve: Strategy vs. Buy & Hold")
ax.set_ylabel("Equity ($)")
ax.legend()
plt.show()

## A note on what this chart can't tell you yet

Buy-and-hold TSLA over a long, mostly-up sample is a genuinely hard benchmark to beat on raw cumulative return — that's expected, and outperforming it on *cumulative return alone* isn't really the bar. What matters more for a trading model are the risk-adjusted numbers (Sharpe/Sortino), drawdown, and trade-level statistics, which Notebook 3 covers in full, along with an honest in-sample/out-of-sample split so the results aren't just a single overfit-prone snapshot.

In [ ]:
equity_curve.to_pickle("equity_curve.pkl")
trade_log.to_pickle("trade_log.pkl")
print("Saved equity_curve.pkl and trade_log.pkl")